# 04. Multimodal fusion: concat + MLP (Phase 4)

Nối vector PhoBERT `<s>` (768) và vector pooled CLIP (768) thành 1536 chiều, qua MLP `1536 → 512 → 4`.
So sánh với baseline một modality **cùng chế độ train**, để chênh lệch đến từ việc kết hợp modality chứ không phải từ việc đóng băng hay fine-tune encoder:

| Config | Modality | PhoBERT | CLIP | lr head / encoder | Vai trò |
|---|---|---|---|---|---|
| `text_only_frozen` | text | đóng băng | — | 1e-3 / — | mốc so sánh |
| `image_only_frozen` | image | — | đóng băng | 1e-3 / — | mốc so sánh |
| `fusion_concat_frozen` | concat | đóng băng | đóng băng | 1e-3 / — | Phase 4: chỉ train MLP |
| `fusion_concat_ftlast` | concat | mở 2 block cuối | mở 2 block cuối + `post_layernorm` | 1e-4 / 2e-5 | Phase 4: fine-tune nhẹ |
| `text_only_frozen_cw` | text | đóng băng | — | 1e-3 / — | mốc so sánh cho class weight |
| `fusion_concat_frozen_reg` | concat | đóng băng | đóng băng | 1e-3 / — | regularize nhánh ảnh |
| `fusion_concat_frozen_reg_cw` | concat | đóng băng | đóng băng | 1e-3 / — | regularize nhánh ảnh + class weight |

Mọi config: tối đa 20 epoch, early stopping patience 3, CE, sampler random, batch 16; không class weight trừ các config `_cw` (`sqrt_inverse`).

**Regularize nhánh ảnh** (`_reg`): kết quả 3 seed đầu cho thấy `fusion_concat_frozen` đúng bằng số mẫu với `text_only_frozen` nhưng train loss giảm nhanh hơn, early stop sớm (khoảng epoch 5 so với 7-14) và tụt ở 2 lớp hiếm: head học thuộc train qua vector CLIP cố định. `_reg` thêm LayerNorm + Linear(256) riêng mỗi nhánh, dropout 0.2 trên vector ảnh, và modality dropout bỏ hẳn vector ảnh ở 30% mẫu train (`model.fusion` trong config).
Mỗi config chạy với 3 seed (`SEEDS`); `data.split_seed` giữ cố định train/val/test (cùng cách chia với seed 42 của Phase 3), nên chênh lệch giữa các seed chỉ đến từ khởi tạo head và thứ tự batch.

**Đọc macro-F1 cẩn thận:** tập test chỉ có 7 mẫu `text-sarcasm`, đúng 3/7 hay 0/7 làm macro-F1 4 lớp lệch khoảng 0.15. Bảng có thêm macro-F1 3 lớp (bỏ `text-sarcasm`) và weighted-F1.

Kaggle: bật **Internet** và **GPU T4 x2** (1 GPU vẫn chạy được, chỉ chậm gấp đôi). Add Input:
1. `hhhoang/vimmsd-dataset`: ảnh và nhãn gốc
2. `funalee/vimmsd-data-preprocessed`: `text_preprocessed.json` (caption đã tách từ, `scripts/train.py` tự copy sang cache)
3. (tuỳ chọn) output của notebook 02 / 03 có thư mục `text_only/`, `image_only/`: bảng so sánh tự thêm baseline fine-tune toàn bộ của Phase 3
4. (tuỳ chọn) output version trước của notebook này: đặt `PREVIOUS_OUTPUT` ở cell cấu hình để dùng lại các run đã xong, chỉ train các run mới

Thời gian ước tính trên 1 GPU: khoảng 10 phút mỗi run (7 config × 3 seed khoảng 3,5 giờ). Với 2 GPU, 2 run chạy song song nên còn khoảng một nửa; dùng lại 12 run cũ thì còn 9 run, khoảng 45 phút.
Output trong `/kaggle/working/outputs/<config>_s<seed>/`: `best.pt`, `test_metrics.json`, `best_val_metrics.json`, `confusion_matrix_test.png`, `test_predictions.csv`, `results_row.md` (thời gian train, peak VRAM).

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "thanhtruc/fusion-concat"  # đổi về "main" sau khi PR Phase 4 được merge; hoặc 1 commit hash để tái lặp
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):  # ảnh Kaggle dựng từ ảnh Colab nên cũng có biến này
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
# Chạy test của repo trước khi train: vài chục giây, model trong test là bản tí hon nên không tải weight.
# Test fail thì dừng notebook luôn, không tốn GPU train code lỗi. Test cần Java/VnCoreNLP tự skip nếu thiếu.
RUN_TESTS = True
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
    result = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-x", "-p", "no:cacheprovider"],
                            capture_output=True, text=True)
    print(result.stdout[-5000:], result.stderr[-2000:])
    if result.returncode != 0:
        raise RuntimeError("pytest fail, xem output phía trên trước khi train")

In [ ]:
import gc
import json
import logging
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from vimmsd.utils.config import load_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

# Bỏ bớt config / seed nếu chỉ muốn chạy lại một phần. Vòng ngoài là seed: bị ngắt giữa chừng vẫn có đủ 4 config của seed đầu.
RUNS = ["text_only_frozen", "image_only_frozen", "fusion_concat_frozen", "fusion_concat_ftlast",
        "text_only_frozen_cw", "fusion_concat_frozen_reg", "fusion_concat_frozen_reg_cw"]
SEEDS = [42, 1, 2]   # 42 = seed của Phase 3
SPLIT_SEED = 42      # cố định train/val/test cho mọi seed
# True: run đã có test_metrics.json và test_predictions.csv thì bỏ qua (attach output version trước rồi chạy tiếp)
SKIP_DONE = True
# Thư mục outputs/ của output version trước (Add Input), vd "/kaggle/input/<tên output notebook 04>/outputs".
# Chạy lại trong cùng session (kết quả đã ở /kaggle/working/outputs) thì để None.
# Các run đã xong được copy sang OUT (bỏ file .pt) để SKIP_DONE bỏ qua. None = không dùng.
PREVIOUS_OUTPUT = None
RARE = "text-sarcasm"  # lớp chỉ có 7 mẫu test, tách riêng khi đọc macro-F1


def run_name(name, seed):
    return f"{name}_s{seed}"


cfgs = {name: load_config(f"configs/{name}.yaml") for name in RUNS}
OUT = Path(next(iter(cfgs.values())).paths.output_dir)
LABELS = list(next(iter(cfgs.values())).data.labels)
print("output:", OUT)
print("labels:", LABELS)
print({name: (cfg.train.epochs, cfg.train.lr_head, cfg.train.lr_encoder) for name, cfg in cfgs.items()})

## Kiểm tra data

Báo lỗi sớm nếu thiếu dataset, thay vì đợi tới lúc train. `stage_preprocessed_cache` là hàm `train.py` cũng gọi: copy `text_preprocessed.json` từ `/kaggle/input` sang `cache_dir` ghi được.

In [ ]:
from vimmsd.data.dataset import stage_preprocessed_cache

cfg0 = next(iter(cfgs.values()))
data_dir = Path(cfg0.paths.data_dir)
if not (data_dir / cfg0.data.train_json).exists():
    raise FileNotFoundError(f"không thấy {data_dir / cfg0.data.train_json}: Add Input hhhoang/vimmsd-dataset")
for path in stage_preprocessed_cache(cfg0):
    meta = json.loads(path.read_text(encoding="utf-8")).get("meta", {})
    print(f"{path.name}: pipeline_key={meta.get('pipeline_key')}, preprocess_version={meta.get('preprocess_version')}")

## Số parameters train được

Build từng model trên CPU để đếm, rồi giải phóng. Dùng cho bảng thống kê tài nguyên.

In [ ]:
from vimmsd.models.classifier import build_model

param_rows = []
for name, cfg in cfgs.items():
    model = build_model(cfg)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    head = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and not n.startswith(("text_encoder.", "image_encoder.")))
    param_rows.append({"config": name, "trainable (M)": trainable / 1e6, "trong đó encoder (M)": (trainable - head) / 1e6,
                       "head (M)": head / 1e6, "tổng (M)": total / 1e6})
    del model
    gc.collect()
params = pd.DataFrame(param_rows).set_index("config")
params.round(2)

## Train + đánh giá

Mỗi config × seed: `train.py` (train, chọn best checkpoint theo val macro-F1, chấm test) rồi `evaluate.py --split test` (ghi `test_predictions.csv` để phân tích lỗi).
Override chung: `seed=<seed> data.split_seed=42 name=<config>_s<seed>`.

**2 GPU:** mỗi run chỉ dùng khoảng 1-1,5 GB VRAM nên không chia một run ra 2 GPU (DDP), mà chạy 2 run song song, mỗi GPU một run (`vimmsd.utils.gpu_queue`, `CUDA_VISIBLE_DEVICES=k`).
Log đầy đủ của từng run nằm ở `outputs/<run>/run.log`; cell chỉ in các dòng epoch khi run xong.

**Dung lượng:** `/kaggle/working` giới hạn khoảng 20 GB. Checkpoint chỉ lưu parameter được train (bỏ encoder đóng băng, tải lại từ pretrained khi load), và `last.pt` bị xoá khi run xong, nên mỗi run đóng băng chỉ tốn vài MB; `fusion_concat_ftlast` khoảng 120 MB.
Hai run dùng chung 4 CPU của Kaggle (mỗi run 2 worker load ảnh) nên `train (phút)` của từng run có thể cao hơn khi chạy một mình; khi so tài nguyên giữa các config, chỉ so các run chạy cùng chế độ.

Session bị ngắt: đặt `PREVIOUS_OUTPUT` trỏ vào output version trước rồi chạy lại cell này, các run đã xong được bỏ qua.

In [ ]:
import shutil

import torch

from vimmsd.utils.gpu_queue import Job, run_on_gpus

NUM_GPUS = torch.cuda.device_count()  # Kaggle T4 x2 -> 2 run song song
print("GPU:", NUM_GPUS, [torch.cuda.get_device_name(k) for k in range(NUM_GPUS)])

if PREVIOUS_OUTPUT and Path(PREVIOUS_OUTPUT).resolve() == OUT.resolve():
    print("PREVIOUS_OUTPUT trùng OUT: kết quả đã nằm sẵn ở đây, không cần copy")
elif PREVIOUS_OUTPUT:
    if not Path(PREVIOUS_OUTPUT).is_dir():
        raise FileNotFoundError(f"không thấy PREVIOUS_OUTPUT={PREVIOUS_OUTPUT}, xem đường dẫn ở panel Input")
    for metrics in sorted(Path(PREVIOUS_OUTPUT).glob("*/test_metrics.json")):
        shutil.copytree(metrics.parent, OUT / metrics.parent.name, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("*.pt"))
        print("dùng lại", metrics.parent.name)


def make_job(name, seed):
    run = run_name(name, seed)
    out_dir = OUT / run
    overrides = [f"seed={seed}", f"data.split_seed={SPLIT_SEED}", f"name={run}"]
    train = [sys.executable, "scripts/train.py", "--config", f"configs/{name}.yaml", "--override", *overrides,
             "--note", f"Phase 4 {name} seed {seed}"]
    evaluate = [sys.executable, "scripts/evaluate.py", "--config", f"configs/{name}.yaml", "--split", "test",
                "--override", *overrides]

    def check():
        missing = [f for f in ("test_metrics.json", "test_predictions.csv") if not (out_dir / f).exists()]
        if missing:
            return f"thiếu {', '.join(missing)}"
        (out_dir / "last.pt").unlink(missing_ok=True)  # chỉ cần để resume giữa run; run xong thì bỏ cho đỡ tốn đĩa
        return None

    return Job(run, [train, evaluate], out_dir / "run.log", check=check)


jobs = []
for seed in SEEDS:  # vòng ngoài là seed: hàng đợi chạy hết seed đầu trước
    for name in RUNS:
        out_dir = OUT / run_name(name, seed)
        if SKIP_DONE and (out_dir / "test_metrics.json").exists() and (out_dir / "test_predictions.csv").exists():
            print(f"=== bỏ qua {run_name(name, seed)}: đã có kết quả trong {out_dir}")
            # run cũ (trước khi checkpoint bỏ encoder đóng băng) còn last.pt gần 1 GB, file .tmp ghi dở khi hết đĩa
            for f in [out_dir / "last.pt", *out_dir.glob("*.tmp")]:
                f.unlink(missing_ok=True)
            continue
        # run chưa xong (lỗi giữa chừng) train lại từ đầu: xoá thư mục cũ, có thể còn checkpoint .tmp ghi dở vài trăm MB
        shutil.rmtree(out_dir, ignore_errors=True)
        jobs.append(make_job(name, seed))
print(f"=== {len(jobs)} run, {max(NUM_GPUS, 1)} run cùng lúc")
OUT.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUT).free / 1e9
print(f"đĩa còn trống: {free_gb:.1f} GB")
if jobs and free_gb < 1:
    raise RuntimeError("đĩa gần đầy: xoá bớt file trong /kaggle/working (!du -sh /kaggle/working/*) trước khi train")

finished = run_on_gpus(jobs, NUM_GPUS, summary_pattern=r"INFO epoch|early stopping|^\| \d{4}-")
print(f"đĩa còn trống: {shutil.disk_usage(OUT).free / 1e9:.1f} GB")
failed = [job for job in finished if job.error]
if failed:
    raise RuntimeError("run lỗi: " + ", ".join(f"{job.name} ({job.error}, log {job.log})" for job in failed))

## Bảng so sánh

Mỗi dòng ở bảng chi tiết là một run (config × seed). Bảng tổng hợp lấy trung bình ± độ lệch chuẩn qua các seed.
- `val`: best val macro-F1 (checkpoint được chọn); `test`: trên tập test chia từ train.
- `macro-F1 (3 lớp)`: bỏ `text-sarcasm` (7 mẫu test), ít nhiễu hơn macro-F1 4 lớp.
- Thời gian, peak VRAM đọc từ `results_row.md`.

Baseline Phase 3 (fine-tune toàn bộ, 1 seed) được thêm vào nếu tìm thấy trong `/kaggle/input`, hoặc điền tay vào `MANUAL_REFERENCE`.

In [ ]:
# Baseline Phase 3 nhập tay khi không attach được output, ví dụ:
# MANUAL_REFERENCE = {"image_only (Phase 3)": {"macro_f1": 0.3945}}
MANUAL_REFERENCE = {}


def parse_results_row(path):
    """| ngày | `config` | `commit` | env | val | test | phút | ghi chú | -> phút, peak VRAM, commit."""
    if not path.exists():
        return {}
    line = path.read_text(encoding="utf-8").strip().splitlines()[-1]  # chạy lại thì dòng mới nhất ở cuối
    cols = [c.strip() for c in line.strip("|").split("|")]  # ghi chú có thể chứa "|" nên chỉ dùng 7 cột đầu
    vram = re.search(r"peak VRAM ([\d.]+) GB", line)
    return {"commit": cols[2].strip("`"), "train (phút)": float(cols[6]),
            "peak VRAM (GB)": float(vram.group(1)) if vram else None}


def metrics_row(config, seed, out_dir, mode):
    test = json.loads((out_dir / "test_metrics.json").read_text())
    val_path = out_dir / "best_val_metrics.json"
    per_class = test["per_class"]
    row = {"config": config, "seed": seed, "chế độ": mode,
           "val macro-F1": json.loads(val_path.read_text())["macro_f1"] if val_path.exists() else np.nan,
           "test macro-F1": test["macro_f1"],
           "test macro-F1 (3 lớp)": np.mean([m["f1"] for c, m in per_class.items() if c != RARE]),
           "test weighted-F1": test["weighted_f1"],
           "test macro-P": test["macro_precision"], "test macro-R": test["macro_recall"], "accuracy": test["accuracy"]}
    row.update({f"F1 {c}": m["f1"] for c, m in per_class.items()})
    row.update(parse_results_row(out_dir / "results_row.md"))
    return row


MODES = {"text_only_frozen": "đóng băng", "image_only_frozen": "đóng băng",
         "fusion_concat_frozen": "đóng băng", "fusion_concat_ftlast": "mở 2 block cuối",
         "text_only_frozen_cw": "đóng băng + class weight", "fusion_concat_frozen_reg": "đóng băng + reg ảnh",
         "fusion_concat_frozen_reg_cw": "đóng băng + reg ảnh + class weight"}
rows = [metrics_row(name, seed, OUT / run_name(name, seed), MODES[name])
        for seed in SEEDS for name in RUNS if (OUT / run_name(name, seed) / "test_metrics.json").exists()]

for ref in ("text_only", "image_only"):
    hits = sorted(Path("/kaggle/input").rglob(f"{ref}/test_metrics.json")) if Path("/kaggle/input").exists() else []
    if hits:
        rows.append(metrics_row(f"{ref} (Phase 3)", 42, hits[0].parent, "fine-tune toàn bộ"))
for label, m in MANUAL_REFERENCE.items():
    rows.append({"config": label, "seed": 42, "chế độ": "fine-tune toàn bộ", "test macro-F1": m.get("macro_f1")})

runs = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
print("Chi tiết từng run:")
display(runs.set_index(["config", "seed"]).round(4))

In [ ]:
METRICS = ["val macro-F1", "test macro-F1", "test macro-F1 (3 lớp)", "test weighted-F1", "accuracy",
           *[f"F1 {c}" for c in LABELS]]
RESOURCES = ["train (phút)", "peak VRAM (GB)"]
order = [c for c in [*RUNS, "text_only (Phase 3)", "image_only (Phase 3)", *MANUAL_REFERENCE] if c in set(runs["config"])]
grouped = runs.groupby("config", sort=False)
mean = grouped[METRICS + RESOURCES].mean().reindex(order)
std = grouped[METRICS].std().reindex(order)  # NaN khi chỉ có 1 seed
n_seeds = grouped.size().reindex(order)

summary = pd.DataFrame(index=order)
summary["chế độ"] = grouped["chế độ"].first().reindex(order)
summary["số seed"] = n_seeds
for col in METRICS:
    summary[col] = [f"{m:.3f}" if np.isnan(s) else f"{m:.3f} ± {s:.3f}" for m, s in zip(mean[col], std[col])]
for col in RESOURCES:
    summary[col] = mean[col].round(2)
summary = summary.join(params[["trainable (M)"]].round(2), how="left")
print("Trung bình ± độ lệch chuẩn qua các seed:")
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
x = np.arange(len(order))
for offset, col in ((-0.2, "test macro-F1"), (0.2, "test macro-F1 (3 lớp)")):
    axes[0].bar(x + offset, mean[col], 0.4, yerr=std[col].fillna(0), capsize=3, label=col)
axes[0].set_xticks(x, order, rotation=20, ha="right")
axes[0].set_ylabel("F1")
axes[0].set_title("Macro-F1 trên test (trung bình ± std qua seed)")
axes[0].legend(fontsize=8)

width = 0.8 / len(order)
for k, config in enumerate(order):
    cols = [f"F1 {c}" for c in LABELS]
    axes[1].bar(np.arange(len(LABELS)) + (k - (len(order) - 1) / 2) * width, mean.loc[config, cols],
                width, yerr=std.loc[config, cols].fillna(0), capsize=2, label=config)
axes[1].set_xticks(np.arange(len(LABELS)), LABELS)
axes[1].set_ylabel("F1")
axes[1].set_title("F1 từng lớp")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Ma trận nhầm lẫn (test, seed đầu tiên)

In [ ]:
done = [name for name in RUNS if (OUT / run_name(name, SEEDS[0]) / "confusion_matrix_test.png").exists()]
fig, axes = plt.subplots(1, len(done), figsize=(5 * len(done), 4.5), squeeze=False)
for ax, name in zip(axes[0], done):
    ax.imshow(plt.imread(OUT / run_name(name, SEEDS[0]) / "confusion_matrix_test.png"))
    ax.set_title(run_name(name, SEEDS[0]))
    ax.axis("off")
plt.tight_layout()
plt.show()

## Ảnh giúp được gì? So sánh từng mẫu: concat với text-only

Mỗi cặp cùng chế độ đóng băng, cùng loss và cùng seed, nên khác biệt chỉ đến từ việc thêm vector ảnh (và cách regularize nó). Config `_cw` so với `text_only_frozen_cw`, còn lại so với `text_only_frozen`. Với mỗi seed, đếm mẫu concat đúng mà text-only sai (ảnh giúp) và ngược lại (ảnh nhiễu) theo nhãn thật, rồi lấy trung bình qua các seed.

In [ ]:
def load_pred(name, seed):
    return pd.read_csv(OUT / run_name(name, seed) / "test_predictions.csv").set_index("id")


HELP, HURT = "ảnh giúp (concat đúng, text sai)", "ảnh nhiễu (text đúng, concat sai)"
examples = {}
PAIRS = [("text_only_frozen", "fusion_concat_frozen"), ("text_only_frozen", "fusion_concat_ftlast"),
         ("text_only_frozen", "fusion_concat_frozen_reg"), ("text_only_frozen_cw", "fusion_concat_frozen_reg_cw")]
for text_name, fusion_name in PAIRS:
    per_seed = []
    for seed in SEEDS:
        if not all((OUT / run_name(n, seed) / "test_predictions.csv").exists() for n in (text_name, fusion_name)):
            continue
        both = load_pred(text_name, seed)[["label", "pred", "caption"]].join(
            load_pred(fusion_name, seed)[["pred"]], rsuffix="_concat")
        text_ok, concat_ok = both["pred"] == both["label"], both["pred_concat"] == both["label"]
        both["text đúng"], both["concat đúng"] = text_ok, concat_ok
        both[HELP], both[HURT] = concat_ok & ~text_ok, text_ok & ~concat_ok
        per_seed.append(both.groupby("label")[["text đúng", "concat đúng", HELP, HURT]].sum())
        examples.setdefault(fusion_name, both)
    if per_seed:
        summary_pairs = sum(per_seed) / len(per_seed)
        summary_pairs.insert(0, "n", examples[fusion_name].groupby("label").size())
        summary_pairs.loc["tổng"] = summary_pairs.sum()
        print(f"=== {text_name} vs {fusion_name} (trung bình {len(per_seed)} seed)")
        display(summary_pairs.round(1))

In [ ]:
# Vài mẫu ảnh giúp ở seed đầu tiên: concat đúng, text-only sai
both = examples.get("fusion_concat_frozen")
if both is not None:
    helped = both[both[HELP]]
    display(helped.sample(min(10, len(helped)), random_state=0)[["label", "pred", "pred_concat", "caption"]])

## Dòng kết quả cho `reports/results.md`

Copy các dòng dưới vào `reports/results.md` rồi commit. Bảng thống kê tài nguyên lấy cột `train (phút)`, `peak VRAM (GB)` và `trainable (M)` ở bảng tổng hợp.

In [ ]:
for seed in SEEDS:
    for name in RUNS:
        path = OUT / run_name(name, seed) / "results_row.md"
        if path.exists():
            print(path.read_text(encoding="utf-8").strip().splitlines()[-1])

## Đọc kết quả

- `fusion_concat_frozen` so với `max(text_only_frozen, image_only_frozen)`: concat có tốt hơn modality mạnh nhất khi chỉ train MLP không. Đây là câu hỏi chính của Phase 4.
- `fusion_concat_ftlast` so với `fusion_concat_frozen`: mở 2 block cuối được thêm bao nhiêu, đổi lại thời gian và VRAM.
- `fusion_concat_frozen_reg` so với `fusion_concat_frozen`: regularize nhánh ảnh có làm train loss giảm chậm lại, train được nhiều epoch hơn và giữ lại 2 lớp hiếm không.
- `fusion_concat_frozen_reg_cw` so với `text_only_frozen_cw`: khi cả hai cùng class weight, ảnh còn thêm được gì; `text_only_frozen_cw` so với `text_only_frozen` cho biết riêng class weight được bao nhiêu.
- Chỉ coi một chênh lệch là thật khi nó lớn hơn độ lệch chuẩn giữa các seed; ưu tiên đọc macro-F1 3 lớp và F1 từng lớp, vì macro-F1 4 lớp bị 7 mẫu `text-sarcasm` chi phối.
- So với Phase 3 (fine-tune toàn bộ): concat đóng băng thấp hơn PhoBERT fine-tune toàn bộ là bình thường; chỉ kết luận về fusion khi so cùng chế độ.